# **Data Preparation & EDA (FORESIGHT dataset)**


**Covering:**

* Data verification
* Data Quality & Cleaning
* Feature Engineering (Profit/Margin)
* Inventory-linked risk analysis
* validation check against the known stockout/slow-mover labels

 ## **Day 01 - Data Verification**

**Loading Libraries**

In [7]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import os

import warnings
warnings.filterwarnings("ignore")


**Loading Dataset**


**1. Loading Dataset By KaggleHub**

In [ ]:
# from collections import defaultdict
# import kagglehub



# path = kagglehub.dataset_download("mrayyanshehzad/synthetic-retail-dataset-10-million-transactions")

# print("Path to dataset files:", path)

# files = os.listdir(path)
# print("\nParent Folder contain files:", files)


# # Update File Name to select which File to Load


# if 'retail_contaminated_dataset' in files:
#     path = os.path.join(path, 'retail_contaminated_dataset')
#     files = os.listdir(path)
#     print("Sub-folder contain files:", files)


# csv_files = []

# for f in files:
#   if f.endswith(".csv"):
#      csv_files.append(f)

# print("\nCSV files in the folder:", csv_files)


# df = {}

# for file in csv_files:
#     file_path = os.path.join(path, file)

#     df_name = file.replace('.csv', '')
#     df[df_name] = pd.read_csv(file_path)
#     print(f"Loaded: {file} as dataframes['{df_name}'] (Shape: {df[df_name].shape})")

In [ ]:
# !mount | grep kaggle

**Dataset loading issue via Kaggle Hub**
* **ISSUE:** kagglehub returns stale/outdated data in Google Colab
* **ROOT CAUSE:** Colab mounts popular Kaggle datasets from its own server-side  read-only NFS filestore (/kaggle/input/<dataset>), pinned to an old dataset version (versions/1), instead of fetching fresh data via the Kaggle API.
* **Confirmed via:** `mount | grep kaggle` -> shows "...versions/1 ... type nfs (ro,...)"
* This NFS mount is READ-ONLY and cannot be deleted/overridden (chmod/sudo rm fail).
* Verified the actual Kaggle server data IS updated (correct files/sizes) via `kaggle datasets files ...` CLI — so this is a Colab-side caching limitation, not a Kaggle backend issue or a code bug.
* FIX: Bypass kagglehub + /kaggle/input entirely; download fresh via Kaggle CLI into local (non-mounted) /content/ storage.

`!kaggle datasets download mrayyanshehzad/synthetic-retail-dataset-10-million-transactions -p /content/fresh_dataset --unzip`

**Checking Server Side Data**

In [ ]:
# # Server side data sending check

# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json

# !kaggle datasets files mrayyanshehzad/synthetic-retail-dataset-10-million-transactions


**2. Loading Dataset By Kaggle CLI**

In [ ]:
# !kaggle datasets download mrayyanshehzad/synthetic-retail-dataset-10-million-transactions -p /content/dataset --unzip

In [ ]:

# base_path = "/content/dataset/retail_contaminated_dataset"  # ya retail_clean_dataset
# files = os.listdir(base_path)
# print("Files:", files)

# csv_files = [f for f in files if f.endswith(".csv")]

# df = {}
# for file in csv_files:
#     file_path = os.path.join(base_path, file)
#     df_name = file.replace('.csv', '')
#     df[df_name] = pd.read_csv(file_path)
#     print(f"Loaded: {file} (Shape: {df[df_name].shape})")

**3. Loading Dataset By Local Memory**

In [8]:
import  zipfile

# base_path = r"F:\Zidio Internship\retail_contaminated_dataset.zip"

# with zipfile.ZipFile(base_path, 'r' ) as zip_ref:
#     zip_ref.extractall(r"F:\Zidio Internship\extracted_files")


files = os.listdir(r"F:\Zidio Internship\extracted_files")
print("Files:", files)

csv_files = [f for f in files if f.endswith(".csv")]

df = {}
for file in csv_files:
    file_path = os.path.join(r"F:\Zidio Internship\extracted_files", file)
    df_name = file.replace('.csv', '')
    df[df_name] = pd.read_csv(file_path)
    print(f"Loaded: {file} (Shape: {df[df_name].shape})")

Files: ['customer_master.csv', 'inventory_snapshot.csv', 'promotions.csv', 'sales_transactions.csv', 'sku_inventory_flags.csv', 'sku_master.csv', 'store_master.csv']
Loaded: customer_master.csv (Shape: (10000, 7))
Loaded: inventory_snapshot.csv (Shape: (26408, 6))
Loaded: promotions.csv (Shape: (100, 8))
Loaded: sales_transactions.csv (Shape: (9945511, 11))
Loaded: sku_inventory_flags.csv (Shape: (600, 6))
Loaded: sku_master.csv (Shape: (5000, 7))
Loaded: store_master.csv (Shape: (30, 5))


**Dataset Loading Flow**

```
📁 /kaggle/input/synthetic-retail-dataset-10-million-transactions (Main Path)
│
├── 📑 README.md
└── 📁 Synthetic Retail Dataset/  <-- [Line: if 'Synthetic Retail Dataset' in files]
    │                                  Enter inside this folder to access .csv files
    ├── 📄 sales_transactions.csv
    ├── 📄 sku_inventory_flags.csv
    ├── 📄 inventory_snapshot.csv
    ├── 📄 store_master.csv
    ├── 📄 customer_master.csv
    ├── 📄 sku_master.csv
    └── 📄 promotions.csv
    
```

**Dataset Relationship**
```
store_master.store_id      ──┐
sku_master.sku_id           ─┼──▶ sales_transactions.csv
customer_master.cust_id     ─┤     (store_id, sku_id, customer_id, promo_id)
promotions.promo_id         ─┘

store_master.store_id   ──┐
sku_master.sku_id        ─┴──▶ inventory_snapshot.csv

sku_master.sku_id  ──▶ sku_inventory_flags.csv   (GROUND-TRUTH labels — do NOT use as a feature,
                                                    only to validate our own risk logic later)
```

### **Data View, Shape & Columns**

In [9]:
for name, table in df.items():
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    print(f" {name}")
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    display(table.head(3))
    print("shape:", table.shape)
    print("columns:", list(table.columns))
    print()

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 customer_master
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------


,cust_id,age,gender,city,loyalty_segment,preferred_channel,registration_date
0,CUST00001,52,Female,Sukkur,Silver,Online,2021-01-01
1,CUST00002,41,Female,Karachi,Silver,Mobile App,2021-05-14
2,CUST00003,48,Female,Bahawalpur,Bronze,In-Store,2022-10-23


shape: (10000, 7)
columns: ['cust_id', 'age', 'gender', 'city', 'loyalty_segment', 'preferred_channel', 'registration_date']

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 inventory_snapshot
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------


,store_id,sku_id,stock_on_hand,reorder_point,safety_stock,last_restock_date
0,ST11,SKU02558,333,72,22,2025-06-23
1,ST21,SKU01031,236,67,14,2025-06-17
2,ST26,SKU02129,496,96,34,2025-07-10


shape: (26408, 6)
columns: ['store_id', 'sku_id', 'stock_on_hand', 'reorder_point', 'safety_stock', 'last_restock_date']

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 promotions
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------


...

shape: (600, 6)
columns: ['sku_id', 'flag', 'affected_stores', 'window_start', 'window_end', 'notes']

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 sku_master
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------


,sku_id,sku_name,category,subcategory,unit_price,cost_price,brand
0,SKU00001,NutriPlus Cookware Large,Home & Kitchen,Cookware,813.41,619.77,NutriPlus
1,SKU00002,CrispKing Bread Family Pack,Dairy & Bakery,Bread,70.38,49.57,CrispKing
2,SKU00003,SoftTouch Notebooks 2L,Stationery & Office,Notebooks,151.28,83.67,SoftTouch


shape: (5000, 7)
columns: ['sku_id', 'sku_name', 'category', 'subcategory', 'unit_price', 'cost_price', 'brand']

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 store_master
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------


,store_id,store_name,city,store_type,opening_date
0,ST01,Quetta Convenience Store #01,Quetta,Convenience Store,2017-07-08
1,ST02,Islamabad Express Store #02,Islamabad,Express Store,2019-05-08
2,ST03,Sialkot Supermarket #03,Sialkot,Supermarket,2019-03-28


shape: (30, 5)
columns: ['store_id', 'store_name', 'city', 'store_type', 'opening_date']



### **Step 1 — Verification (before manipulating data)**


**Primary Data Verification**

In [10]:
expected_rows = {
    "store_master": 30,
    "sku_master": 5000,
    'customer_master': 10000,
    'promotions': 100,
}

for name, expected in expected_rows.items():
    actual = len(df[name])
    # status = "OK" if actual == expected else "MISMATCH"
    status = "Unknown"
    if actual == expected:
      status = "OK"
      print(f"{name:20s}   actual={actual:>7}   expected={expected:>7}   [{status}]")
    elif actual > expected:
      status = "EXCESS"
      print(f"{name:20s}   actual={actual:>7}   expected={expected:>7}  [{status}]")
    else:
      status = "MISMATCH"
      print(f"{name:20s}   actual={actual:>7}   expected={expected:>7}  [{status}]")

store_master           actual=     30   expected=     30   [OK]
sku_master             actual=   5000   expected=   5000   [OK]
customer_master        actual=  10000   expected=  10000   [OK]
promotions             actual=    100   expected=    100   [OK]


**Secondary Data Verification**

In [11]:
print(f"{'inventory_snapshot':20s}   actual={len(df['inventory_snapshot']):>8}   expected= no fixed target (quick-check only)")
print(f"{'sales_transactions':20s}   actual={len(df['sales_transactions']):>8}   expected= 10 Million")
print(f"{'sku_inventory_flags':20s}   actual={len(df['sku_inventory_flags']):>8}   expected= 600 (200 STOCKOUT_RISK + 400 SLOW_MOVER)")


inventory_snapshot     actual=   26408   expected= no fixed target (quick-check only)
sales_transactions     actual= 9945511   expected= 10 Million
sku_inventory_flags    actual=     600   expected= 600 (200 STOCKOUT_RISK + 400 SLOW_MOVER)


In [12]:
print(df['sku_inventory_flags']['flag'].value_counts())

flag
SLOW_MOVER       400
STOCKOUT_RISK    200
Name: count, dtype: int64


**Master tables `primary key` uniqueness check**

In [13]:
primary_key = {
    "store_master": "store_id",
    "sku_master": "sku_id",
    'customer_master': "cust_id",
    'promotions': "promo_id",
}

for name, key in primary_key.items():
    duplicates = df[name][key].duplicated().sum()
    print(f"{name:20s}   duplicates={duplicates:>3}")



store_master           duplicates=  0
sku_master             duplicates=  0
customer_master        duplicates=  0
promotions             duplicates=  0


**Referential integrity**
 > every FK in sales/inventory must exist in its master table

In [14]:
checks = [
    #(child table) (child key) (parent table) (parent key)
    ("sales_transactions", "store_id", "store_master", "store_id"),
    ("sales_transactions", "sku_id", "sku_master", "sku_id"),
    ("sales_transactions", "customer_id", "customer_master", "cust_id"),
    ("inventory_snapshot", "store_id", "store_master", "store_id"),
    ("inventory_snapshot", "sku_id", "sku_master", "sku_id"),
]

for child_table, child_key, parent_table, parent_key in checks:

  child_values = df[child_table][child_key]
  parent_values = df[parent_table][parent_key]

  valid = child_values.isin(parent_values).all()

  # print(f"{child_table}.{child_key:13s} -> \t {parent_table}.{parent_key}: {'OK' if valid else 'MISMATCH':}")

  child_full = f"{child_table}.{child_key}"
  parent_full = f"{parent_table}.{parent_key}"
  status = 'OK' if valid else 'MISMATCH'

  print(f"{child_full:32s} ->    {parent_full:26s} :    {status}")


sales_transactions.store_id      ->    store_master.store_id      :    OK
sales_transactions.sku_id        ->    sku_master.sku_id          :    OK
sales_transactions.customer_id   ->    customer_master.cust_id    :    OK
inventory_snapshot.store_id      ->    store_master.store_id      :    OK
inventory_snapshot.sku_id        ->    sku_master.sku_id          :    OK


`promo_id` is allowed to be null (no promotion applied) - check only the non-null ones

In [15]:
promo_values = df['sales_transactions']['promo_id'].dropna()

valid = promo_values.isin(df['promotions']['promo_id']).all()
print(f"sales_transactions.promo_id (non-null) -> promotions.promo_id: {'OK' if valid else 'MISMATCH'}")


sales_transactions.promo_id (non-null) -> promotions.promo_id: OK


## **Day 02 -  Data Quality & Cleaning**

**Also missing before:** an explicit null audit, duplicate check, value-range sanity checks, and a math-consistency check on `total_value`. This is the actual D1/D2 deliverable work — not just `.info()`.

**Get Null Count**

In [16]:
for name, table in df.items():
  nulls = table.isnull().sum()
  nulls = nulls[nulls > 0]


  print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
  print(f" {name}")
  print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------")

  if len(nulls):
    print(nulls)
  else:
    print("no nulls")

  # print(nulls if len(nulls) else "no nulls")


---------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 customer_master
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------
no nulls
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 inventory_snapshot
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------
no nulls
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 promotions
-------------------------------------------------------------------------

**Insight:**
* `promo_id` is null on most `sales_transactions` rows (as no promo applied - so it's not an error)
* `window_start`/`window_end` is null on all `SLOW_MOVER` rows in `sku_inventory_flags` (as those columns only apply to `STOCKOUT_RISK`).

So I'll Fill/flag them explicitly rather than leaving raw NaN:

In [18]:
df['sales_transactions'].head()

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id
0,2025-04-02,RCPT00000001,ST16,SKU02498,CUST01410,1,2379.11,2379.11,In-Store,0.0,NaN
1,2025-04-02,RCPT00000001,ST16,SKU04596,CUST01410,3,335.55,1006.65,In-Store,0.0,NaN
2,2022-04-24,RCPT00000002,ST15,SKU00078,CUST00134,1,820.53,820.53,Online,0.0,NaN
3,2022-04-24,RCPT00000002,ST15,SKU00554,CUST00134,2,88.32,176.64,Online,0.0,NaN
4,2024-09-22,RCPT00000003,ST20,SKU03727,CUST08826,1,1660.93,1660.93,Online,0.0,NaN


In [19]:
df['sales_transactions']['promo_id'].fillna("NO_PROMO", inplace=True)

df['sales_transactions']['promo_applied'] = df['sales_transactions']['promo_id'] != "NO_PROMO"

In [20]:
df['sku_inventory_flags']['window_start'].isna().sum()

np.int64(400)

In [21]:

# df['sku_inventory_flags']['window_start'].fillna(0, inplace=True)
# df['sku_inventory_flags']['window_end'].fillna(0, inplace=True)

**Duplicate rows**

In [22]:
df['sales_transactions'][df['sales_transactions'].duplicated()]


,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id,promo_applied
272,2025-05-11,RCPT00000149,ST10,SKU02141,CUST09365,2,471.88,943.76,Online,0.0,NaN,True
315,2022-08-03,RCPT00000169,ST29,SKU04321,CUST07066,1,961.30,961.30,In-Store,0.0,NaN,True
545,2025-11-14,RCPT00000289,ST06,SKU04321,CUST02146,1,961.30,961.30,In-Store,0.0,NaN,True
2936,2024-06-22,RCPT00001531,ST25,SKU04321,CUST01969,1,961.30,744.05,Online,22.6,PROMO045,True
5846,2025-10-09,RCPT00003040,ST13,SKU04321,CUST08409,1,961.30,961.30,Mobile App,0.0,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...
9941292,2024-10-09,RCPT05179166,ST14,SKU04321,CUST08610,1,961.30,961.30,In-Store,0.0,NaN,True
9941408,2023-11-16,RCPT05179233,ST14,SKU04321,CUST00802,1,961.30,961.30,In-Store,0.0,NaN,True
9942163,2023-02-15,RCPT05179621,ST08,SKU04321,CUST04023,1,961.30,961.30,Online,0.0,NaN,True
9943995,2023-08-05,RCPT05180578,ST14,SKU04321,CUST02275,1,961.30,722.90,In-Store,24.8,PROMO061,True


In [23]:
duplicates_rows = df['sales_transactions'].duplicated().sum()
print("Duplicated rows in sales_transaction:",duplicates_rows)

Duplicated rows in sales_transaction: 12897


In [25]:
duplicate_id = df['sales_transactions'].duplicated(subset=['receipt_id', 'sku_id']).sum()
print("Duplicate (receipt_id, sku_id) combos:", duplicate_id)

Duplicate (receipt_id, sku_id) combos: 37206


**Insight:** Can be valid but same item added twice on one receipt

**Value-range  checks**

> Avoid negative values

In [27]:
quantity = (df['sales_transactions']['quantity'] <= 0).sum()
print("quantity <= 0:", quantity)

unit_price = (df['sales_transactions']['unit_price'] <= 0).sum()
print("unit_price <= 0:", unit_price)

discount_pct = (~df['sales_transactions']['discount_pct'].between(0, 100)).sum()
print("discount_pct out of [0,100]:", discount_pct)



stock_on_hand = (df['inventory_snapshot']['stock_on_hand'] < 0).sum()
print("stock_on_hand < 0:", stock_on_hand)

reorder_point = (df['inventory_snapshot']['reorder_point'] < 0).sum()
print("reorder_point < 0:", (df['inventory_snapshot']['reorder_point'] < 0).sum())


age = (~df['customer_master']['age'].between(10, 100)).sum()
print("age out of [10,100]:", age)

quantity <= 0: 0
unit_price <= 0: 0
discount_pct out of [0,100]: 0
stock_on_hand < 0: 0
reorder_point < 0: 0
age out of [10,100]: 0


**Insight:** There exist no out of bound value.

**Why i choose only these columns to check value-range?**

Reason being selecting only these specific columns is that these are the tables which contains numeric values in whole dataset , rest are either strings or objects

In [28]:
for name, table in df.items():
  print("-----------------------------------------------------------------------------------------------------------------")
  print(name)
  print("-----------------------------------------------------------------------------------------------------------------")
  table.info()

-----------------------------------------------------------------------------------------------------------------
customer_master
-----------------------------------------------------------------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   cust_id            10000 non-null  str  
 1   age                10000 non-null  int64
 2   gender             10000 non-null  str  
 3   city               10000 non-null  str  
 4   loyalty_segment    10000 non-null  str  
 5   preferred_channel  10000 non-null  str  
 6   registration_date  10000 non-null  str  
dtypes: int64(1), str(6)
memory usage: 990.8 KB
-----------------------------------------------------------------------------------------------------------------
inventory_snapshot
-----------------------------------------------------------------

**Math Consistency Check**

> Does `total_value` logically correct with `quantity` × `price` × `discount`?

In [29]:
# Copy a sample data
sample = df['sales_transactions'].sample(2000, random_state=42).copy()

# Expected values
sample['expected_total'] = sample['quantity'] * sample['unit_price'] * (1 - sample['discount_pct'] / 100)

# Difference between actual & expected
sample['diff'] = (sample['expected_total'] - sample['total_value']).abs()

print("Rows with > Rupees:1 mismatch:", (sample['diff'] > 1).sum(), "out of", len(sample))

sample[sample['diff'] > 1].head()


Rows with > Rupees:1 mismatch: 0 out of 2000


,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id,promo_applied,expected_total,diff


**Insight:** our data is mathematically correct.

## **Day 03 - Feature Engineering**

**Calendar features**

Extract Days, Months , Quarter & Years as separate features to better visualize data later on.

>Fixed to sort in real calendar order.

In [ ]:
# df['sales_transactions']['months'] =df['sales_transactions']['date'].apply(extract)
# df['sales_transactions'].head()

In [ ]:
# def extract(txt):
#   data = txt.split("-")[0]
#   data = int(data)
#   return data

In [ ]:
# df['sales_transactions']['years'] = df['sales_transactions']['date'].apply(extract)

In [30]:
df['sales_transactions']['date'].head()

0    2025-04-02
1    2025-04-02
2    2022-04-24
3    2022-04-24
4    2024-09-22
Name: date, dtype: str

In [31]:
df['sales_transactions']['date'] = pd.to_datetime(df['sales_transactions']['date'])


In [32]:
sales = df['sales_transactions']

In [33]:
sales['year'] = sales['date'].dt.year

In [34]:
sales['month_num'] = sales['date'].dt.month
sales

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id,promo_applied,year,month_num
0,2025-04-02,RCPT00000001,ST16,SKU02498,CUST01410,1,2379.11,2379.11,In-Store,0.0,NaN,True,2025,4
1,2025-04-02,RCPT00000001,ST16,SKU04596,CUST01410,3,335.55,1006.65,In-Store,0.0,NaN,True,2025,4
2,2022-04-24,RCPT00000002,ST15,SKU00078,CUST00134,1,820.53,820.53,Online,0.0,NaN,True,2022,4
3,2022-04-24,RCPT00000002,ST15,SKU00554,CUST00134,2,88.32,176.64,Online,0.0,NaN,True,2022,4
4,2024-09-22,RCPT00000003,ST20,SKU03727,CUST08826,1,1660.93,1660.93,Online,0.0,NaN,True,2024,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9945506,2023-09-09,RCPT05181381,ST04,SKU00701,CUST06397,2,39.24,68.51,In-Store,12.7,PROMO010,True,2023,9
9945507,2025-12-03,RCPT05181382,ST03,SKU02163,CUST04945,1,791.70,791.70,In-Store,0.0,NaN,True,2025,12
9945508,2025-12-03,RCPT05181382,ST03,SKU02866,CUST04945,4,94.26,377.04,In-Store,0.0,NaN,True,2025,12
9945509,2023-01-20,RCPT05181383,ST03,SKU03727,CUST05244,1,1660.93,1660.93,In-Store,0.0,NaN,True,2023,1


In [35]:
# def extract(txt):
#   months = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun', 7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'}
#   month_num = txt
#   if month_num in months:
#     data = months[month_num]
#   return data

In [39]:
# sales['month'] = sales['month_num'].apply(extract)

Do the same as above 2 cell code, `strftime` : string format time, `%b` : return short names

In [37]:
sales['month'] = sales['date'].dt.strftime('%b')

# Defining the order of months
sales['month'] = pd.Categorical(sales['month'], categories=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], ordered=True)

In [38]:
sales['quarter'] = sales['date'].dt.quarter

In [40]:
sales['day'] = sales['date'].dt.day_name()

**AI Guide:** Pandas mein `.dt.dayofweek` har date ke liye hafte ka din nikalta hai, lekin yeh humein naam (Monday, Tuesday) nahi deta, balkay 0 se 6 tak ek number deta hai.

> Hafte ke aakhri do din (Saturday aur Sunday) ke numbers 5 aur 6 hain

In [41]:
sales['is_weekend'] = sales['date'].dt.dayofweek >= 5

In [42]:

sales.sort_values(by='date', inplace=True)
sales.head()

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id,promo_applied,year,month_num,month,quarter,day,is_weekend
332806,2022-01-01,RCPT00173600,ST21,SKU02449,CUST03000,3,137.83,413.49,In-Store,0.0,NaN,True,2022,1,Jan,1,Saturday,True
6127406,2022-01-01,RCPT03192172,ST03,SKU01246,CUST04900,1,526.86,526.86,In-Store,0.0,NaN,True,2022,1,Jan,1,Saturday,True
7584971,2022-01-01,RCPT03950898,ST07,SKU02895,CUST01180,1,119.21,119.21,In-Store,0.0,NaN,True,2022,1,Jan,1,Saturday,True
304787,2022-01-01,RCPT00159016,ST30,SKU03149,CUST05466,2,610.58,1221.16,Online,0.0,NaN,True,2022,1,Jan,1,Saturday,True
7172382,2022-01-01,RCPT03735831,ST06,SKU01276,CUST04086,2,1990.24,3980.48,In-Store,0.0,NaN,True,2022,1,Jan,1,Saturday,True


**Margin**

This was completely missing before. `sku_master.cost_price` was loaded and never used, so no rupee-impact analysis was possible.

In [43]:
df['sku_master'].head()

,sku_id,sku_name,category,subcategory,unit_price,cost_price,brand
0,SKU00001,NutriPlus Cookware Large,Home & Kitchen,Cookware,813.41,619.77,NutriPlus
1,SKU00002,CrispKing Bread Family Pack,Dairy & Bakery,Bread,70.38,49.57,CrispKing
2,SKU00003,SoftTouch Notebooks 2L,Stationery & Office,Notebooks,151.28,83.67,SoftTouch
3,SKU00004,SunriseFoods Eggs Large,Dairy & Bakery,Eggs,233.64,156.32,SunriseFoods
4,SKU00005,SunriseFoods Pest Control Pack of 6,Home Care,Pest Control,138.50,83.46,SunriseFoods


In [44]:
sku_margin = df['sku_master'][['sku_id', 'unit_price', 'cost_price']].copy()

In [45]:
sku_margin['unit_margin'] = sku_margin['unit_price'] - sku_margin['cost_price']

In [46]:
sku_margin['margin_pct'] = (sku_margin['unit_margin'] / sku_margin['unit_price'] * 100).round(1)

In [47]:
sku_margin.head()

,sku_id,unit_price,cost_price,unit_margin,margin_pct
0,SKU00001,813.41,619.77,193.64,23.8
1,SKU00002,70.38,49.57,20.81,29.6
2,SKU00003,151.28,83.67,67.61,44.7
3,SKU00004,233.64,156.32,77.32,33.1
4,SKU00005,138.50,83.46,55.04,39.7


In [48]:
sales = sales.merge(sku_margin[['sku_id', 'cost_price', 'unit_margin', 'margin_pct']], on='sku_id', how='left')

In [49]:
sales.head()

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,...,promo_applied,year,month_num,month,quarter,day,is_weekend,cost_price,unit_margin,margin_pct
0,2022-01-01,RCPT00173600,ST21,SKU02449,CUST03000,3,137.83,413.49,In-Store,0.0,...,True,2022,1,Jan,1,Saturday,True,99.61,38.22,27.7
1,2022-01-01,RCPT03192172,ST03,SKU01246,CUST04900,1,526.86,526.86,In-Store,0.0,...,True,2022,1,Jan,1,Saturday,True,364.68,162.18,30.8
2,2022-01-01,RCPT03950898,ST07,SKU02895,CUST01180,1,119.21,119.21,In-Store,0.0,...,True,2022,1,Jan,1,Saturday,True,69.15,50.06,42.0
3,2022-01-01,RCPT00159016,ST30,SKU03149,CUST05466,2,610.58,1221.16,Online,0.0,...,True,2022,1,Jan,1,Saturday,True,401.94,208.64,34.2
4,2022-01-01,RCPT03735831,ST06,SKU01276,CUST04086,2,1990.24,3980.48,In-Store,0.0,...,True,2022,1,Jan,1,Saturday,True,1395.07,595.17,29.9


**Profit**

In [50]:
sales['total_cost'] = sales['quantity'] * sales['cost_price']

In [51]:
sales['total_profit'] = sales['total_value'] - sales['total_cost']

In [52]:
sales['profit_pct'] = (sales['total_profit'] / sales['total_value'] * 100).round(2)

In [53]:
sales[['sku_id', 'quantity', 'total_value', 'total_cost', 'total_profit', 'profit_pct']].head()

,sku_id,quantity,total_value,total_cost,total_profit,profit_pct
0,SKU02449,3,413.49,298.83,114.66,27.73
1,SKU01246,1,526.86,364.68,162.18,30.78
2,SKU02895,1,119.21,69.15,50.06,41.99
3,SKU03149,2,1221.16,803.88,417.28,34.17
4,SKU01276,2,3980.48,2790.14,1190.34,29.90


**Redefine the `sales_transaction` dataset**

In [54]:
df['sales_transactions'] = sales

In [55]:
df['sales_transactions'].head()

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,...,month,quarter,day,is_weekend,cost_price,unit_margin,margin_pct,total_cost,total_profit,profit_pct
0,2022-01-01,RCPT00173600,ST21,SKU02449,CUST03000,3,137.83,413.49,In-Store,0.0,...,Jan,1,Saturday,True,99.61,38.22,27.7,298.83,114.66,27.73
1,2022-01-01,RCPT03192172,ST03,SKU01246,CUST04900,1,526.86,526.86,In-Store,0.0,...,Jan,1,Saturday,True,364.68,162.18,30.8,364.68,162.18,30.78
2,2022-01-01,RCPT03950898,ST07,SKU02895,CUST01180,1,119.21,119.21,In-Store,0.0,...,Jan,1,Saturday,True,69.15,50.06,42.0,69.15,50.06,41.99
3,2022-01-01,RCPT00159016,ST30,SKU03149,CUST05466,2,610.58,1221.16,Online,0.0,...,Jan,1,Saturday,True,401.94,208.64,34.2,803.88,417.28,34.17
4,2022-01-01,RCPT03735831,ST06,SKU01276,CUST04086,2,1990.24,3980.48,In-Store,0.0,...,Jan,1,Saturday,True,1395.07,595.17,29.9,2790.14,1190.34,29.90


**Products (aka SKUs):**

Become easy for analysis, instead of going merging indirectly to the parent dataset

In [56]:
df['sku_master'].head()

,sku_id,sku_name,category,subcategory,unit_price,cost_price,brand
0,SKU00001,NutriPlus Cookware Large,Home & Kitchen,Cookware,813.41,619.77,NutriPlus
1,SKU00002,CrispKing Bread Family Pack,Dairy & Bakery,Bread,70.38,49.57,CrispKing
2,SKU00003,SoftTouch Notebooks 2L,Stationery & Office,Notebooks,151.28,83.67,SoftTouch
3,SKU00004,SunriseFoods Eggs Large,Dairy & Bakery,Eggs,233.64,156.32,SunriseFoods
4,SKU00005,SunriseFoods Pest Control Pack of 6,Home Care,Pest Control,138.50,83.46,SunriseFoods


In [57]:
products_data = df['sku_master'][['sku_id','sku_name','category',	'subcategory']].copy()

In [58]:
products_data

,sku_id,sku_name,category,subcategory
0,SKU00001,NutriPlus Cookware Large,Home & Kitchen,Cookware
1,SKU00002,CrispKing Bread Family Pack,Dairy & Bakery,Bread
2,SKU00003,SoftTouch Notebooks 2L,Stationery & Office,Notebooks
3,SKU00004,SunriseFoods Eggs Large,Dairy & Bakery,Eggs
4,SKU00005,SunriseFoods Pest Control Pack of 6,Home Care,Pest Control
...,...,...,...,...
4995,SKU04996,DailyBest Women's Wear 2kg,Apparel & Footwear,Women's Wear
4996,SKU04997,CrispKing Pest Control 250ml,Home Care,Pest Control
4997,SKU04998,CleanWave Kids Wear Medium,Apparel & Footwear,Kids Wear
4998,SKU04999,NovaFresh Oral Care 1L,Personal Care,Oral Care


In [59]:
products = df['sales_transactions'].merge(products_data, on='sku_id', how='left')

In [60]:
products.head()

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,...,is_weekend,cost_price,unit_margin,margin_pct,total_cost,total_profit,profit_pct,sku_name,category,subcategory
0,2022-01-01,RCPT00173600,ST21,SKU02449,CUST03000,3,137.83,413.49,In-Store,0.0,...,True,99.61,38.22,27.7,298.83,114.66,27.73,EverydayPlus Notebooks 2kg,Stationery & Office,Notebooks
1,2022-01-01,RCPT03192172,ST03,SKU01246,CUST04900,1,526.86,526.86,In-Store,0.0,...,True,364.68,162.18,30.8,364.68,162.18,30.78,SmartBuy Vitamins & Supplements 1kg,Health & Wellness,Vitamins & Supplements
2,2022-01-01,RCPT03950898,ST07,SKU02895,CUST01180,1,119.21,119.21,In-Store,0.0,...,True,69.15,50.06,42.0,69.15,50.06,41.99,CleanWave Chocolates Pack of 6,Snacks & Confectionery,Chocolates
3,2022-01-01,RCPT00159016,ST30,SKU03149,CUST05466,2,610.58,1221.16,Online,0.0,...,True,401.94,208.64,34.2,803.88,417.28,34.17,HomeStyle Frozen Vegetables 1kg,Frozen Foods,Frozen Vegetables
4,2022-01-01,RCPT03735831,ST06,SKU01276,CUST04086,2,1990.24,3980.48,In-Store,0.0,...,True,1395.07,595.17,29.9,2790.14,1190.34,29.90,CityFresh Footwear Pack of 12,Apparel & Footwear,Footwear


In [61]:
products.isna().sum()

date                   0
receipt_id             0
store_id               0
sku_id                 0
customer_id            0
quantity               0
unit_price             0
total_value            0
channel                0
discount_pct           0
promo_id         7855345
promo_applied          0
year                   0
month_num              0
month                  0
quarter                0
day                    0
is_weekend             0
cost_price             0
unit_margin            0
margin_pct             0
total_cost             0
total_profit           0
profit_pct             0
sku_name               0
category               0
subcategory            0
dtype: int64

In [63]:
df['sales_transactions'] = products

In [64]:
df['sales_transactions']

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,...,is_weekend,cost_price,unit_margin,margin_pct,total_cost,total_profit,profit_pct,sku_name,category,subcategory
0,2022-01-01,RCPT00173600,ST21,SKU02449,CUST03000,3,137.83,413.49,In-Store,0.0,...,True,99.61,38.22,27.7,298.83,114.66,27.73,EverydayPlus Notebooks 2kg,Stationery & Office,Notebooks
1,2022-01-01,RCPT03192172,ST03,SKU01246,CUST04900,1,526.86,526.86,In-Store,0.0,...,True,364.68,162.18,30.8,364.68,162.18,30.78,SmartBuy Vitamins & Supplements 1kg,Health & Wellness,Vitamins & Supplements
2,2022-01-01,RCPT03950898,ST07,SKU02895,CUST01180,1,119.21,119.21,In-Store,0.0,...,True,69.15,50.06,42.0,69.15,50.06,41.99,CleanWave Chocolates Pack of 6,Snacks & Confectionery,Chocolates
3,2022-01-01,RCPT00159016,ST30,SKU03149,CUST05466,2,610.58,1221.16,Online,0.0,...,True,401.94,208.64,34.2,803.88,417.28,34.17,HomeStyle Frozen Vegetables 1kg,Frozen Foods,Frozen Vegetables
4,2022-01-01,RCPT03735831,ST06,SKU01276,CUST04086,2,1990.24,3980.48,In-Store,0.0,...,True,1395.07,595.17,29.9,2790.14,1190.34,29.90,CityFresh Footwear Pack of 12,Apparel & Footwear,Footwear
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9945506,2025-12-31,RCPT01754437,ST27,SKU03049,CUST03640,2,267.40,534.80,In-Store,0.0,...,False,182.70,84.70,31.7,365.40,169.40,31.68,DailyBest Writing Instruments 2kg,Stationery & Office,Writing Instruments
9945507,2025-12-31,RCPT02617294,ST02,SKU04321,CUST05487,3,961.30,2883.90,Mobile App,0.0,...,False,658.32,302.98,31.5,1974.96,908.94,31.52,EliteHome Hair Care Pack of 6,Personal Care,Hair Care
9945508,2025-12-31,RCPT01754437,ST27,SKU01706,CUST03640,2,1068.65,2137.30,In-Store,0.0,...,False,824.51,244.14,22.8,1649.02,488.28,22.85,BasicNeeds Home Decor 1kg,Home & Kitchen,Home Decor
9945509,2025-12-31,RCPT01537445,ST11,SKU02457,CUST04108,2,822.17,1644.34,In-Store,0.0,...,False,631.31,190.86,23.2,1262.62,381.72,23.21,NutriPlus Ice Cream Family Pack,Frozen Foods,Ice Cream


**Customer**

Same it becomes convienet to analyze data of sales with respect to customer  

In [65]:
df['customer_master']

,cust_id,age,gender,city,loyalty_segment,preferred_channel,registration_date
0,CUST00001,52,Female,Sukkur,Silver,Online,2021-01-01
1,CUST00002,41,Female,Karachi,Silver,Mobile App,2021-05-14
2,CUST00003,48,Female,Bahawalpur,Bronze,In-Store,2022-10-23
3,CUST00004,45,Male,Karachi,Silver,In-Store,2015-07-15
4,CUST00005,39,Female,Faisalabad,Bronze,In-Store,2024-01-02
...,...,...,...,...,...,...,...
9995,CUST09996,59,Female,Karachi,Gold,In-Store,2025-05-10
9996,CUST09997,30,Male,Lahore,Gold,Online,2024-09-30
9997,CUST09998,36,Male,Bahawalpur,Bronze,In-Store,2020-03-31
9998,CUST09999,39,Female,Rawalpindi,Bronze,In-Store,2018-01-14


In [66]:
customer_data = df['customer_master'][['cust_id',	'age'	,'gender'	,'city']].rename(columns={'cust_id': 'customer_id'})

In [67]:
customer_data.head()

,customer_id,age,gender,city
0,CUST00001,52,Female,Sukkur
1,CUST00002,41,Female,Karachi
2,CUST00003,48,Female,Bahawalpur
3,CUST00004,45,Male,Karachi
4,CUST00005,39,Female,Faisalabad


In [68]:
customer = df['sales_transactions'].merge(customer_data, on='customer_id', how='left')

In [69]:
customer.head()

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,...,margin_pct,total_cost,total_profit,profit_pct,sku_name,category,subcategory,age,gender,city
0,2022-01-01,RCPT00173600,ST21,SKU02449,CUST03000,3,137.83,413.49,In-Store,0.0,...,27.7,298.83,114.66,27.73,EverydayPlus Notebooks 2kg,Stationery & Office,Notebooks,48,Male,Quetta
1,2022-01-01,RCPT03192172,ST03,SKU01246,CUST04900,1,526.86,526.86,In-Store,0.0,...,30.8,364.68,162.18,30.78,SmartBuy Vitamins & Supplements 1kg,Health & Wellness,Vitamins & Supplements,43,Male,Karachi
2,2022-01-01,RCPT03950898,ST07,SKU02895,CUST01180,1,119.21,119.21,In-Store,0.0,...,42.0,69.15,50.06,41.99,CleanWave Chocolates Pack of 6,Snacks & Confectionery,Chocolates,36,Male,Islamabad
3,2022-01-01,RCPT00159016,ST30,SKU03149,CUST05466,2,610.58,1221.16,Online,0.0,...,34.2,803.88,417.28,34.17,HomeStyle Frozen Vegetables 1kg,Frozen Foods,Frozen Vegetables,35,Male,Gujranwala
4,2022-01-01,RCPT03735831,ST06,SKU01276,CUST04086,2,1990.24,3980.48,In-Store,0.0,...,29.9,2790.14,1190.34,29.90,CityFresh Footwear Pack of 12,Apparel & Footwear,Footwear,18,Female,Rawalpindi


In [70]:
df['sales_transactions'] = customer

In [71]:
df['sales_transactions'].head()

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,...,margin_pct,total_cost,total_profit,profit_pct,sku_name,category,subcategory,age,gender,city
0,2022-01-01,RCPT00173600,ST21,SKU02449,CUST03000,3,137.83,413.49,In-Store,0.0,...,27.7,298.83,114.66,27.73,EverydayPlus Notebooks 2kg,Stationery & Office,Notebooks,48,Male,Quetta
1,2022-01-01,RCPT03192172,ST03,SKU01246,CUST04900,1,526.86,526.86,In-Store,0.0,...,30.8,364.68,162.18,30.78,SmartBuy Vitamins & Supplements 1kg,Health & Wellness,Vitamins & Supplements,43,Male,Karachi
2,2022-01-01,RCPT03950898,ST07,SKU02895,CUST01180,1,119.21,119.21,In-Store,0.0,...,42.0,69.15,50.06,41.99,CleanWave Chocolates Pack of 6,Snacks & Confectionery,Chocolates,36,Male,Islamabad
3,2022-01-01,RCPT00159016,ST30,SKU03149,CUST05466,2,610.58,1221.16,Online,0.0,...,34.2,803.88,417.28,34.17,HomeStyle Frozen Vegetables 1kg,Frozen Foods,Frozen Vegetables,35,Male,Gujranwala
4,2022-01-01,RCPT03735831,ST06,SKU01276,CUST04086,2,1990.24,3980.48,In-Store,0.0,...,29.9,2790.14,1190.34,29.90,CityFresh Footwear Pack of 12,Apparel & Footwear,Footwear,18,Female,Rawalpindi


## **Checkpointing/Staging Data**

Saving `sales_transaction` as new: file to reduce processor load.

**Mount G-Drive:** In order to store files in google drive for later use.

In [72]:
# from google.colab import drive
# drive.mount('/content/drive')

**1. Standard CSV Format**

In [73]:
# df['sales_transactions'].to_csv("Datasets/processed_sales_transaction", index=False)

**2. Parquet Format (The Smart & Fast Way)**

> Because it support Compression & Decompression

In [74]:
# df['sales_transactions'].to_parquet('/content/drive/MyDrive/Zidio/processed_sales_transactions.parquet')

df['sales_transactions'].to_parquet('Datasets/extracted_files/processed_sales_transactions.parquet')